# 02 - TF-IDF Phase 1 Benchmark

This notebook is an analysis/recipe notebook for the TF-IDF extractive baseline.

Canonical official artifact generation is handled by script: `scripts/generate_official_validation_artifacts.py`.

Objectives:
- evaluate the TF-IDF baseline on the processed VietNews dataset under the frozen Phase 0 protocol
- compare multiple `top_k` settings on the same fixed subset
- inspect and explain benchmark outputs in notebook form


## 1. Environment Setup

This notebook is intentionally restricted to `data/processed/vietnews/`.
It does not fall back to raw files and does not perform API smoke tests, so the benchmark remains protocol-aligned and reproducible.

In [57]:
from __future__ import annotations

import json
import platform
import random
import subprocess
import sys
import time
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "backend").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "backend").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Repository root not found. Open the notebook from the repo root or notebooks/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.services.input import process_from_text
from app.services.summarization.tfidf_summarizer import summarize_with_tfidf
from evaluation.evaluator import Evaluator

pd.set_option("display.max_colwidth", 240)
evaluator = Evaluator(use_stemmer=False)


def safe_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def safe_git_commit(repo_root: Path) -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


def is_git_dirty(repo_root: Path) -> bool | None:
    try:
        result = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )
        return bool(result.stdout.strip())
    except Exception:
        return None


environment_snapshot = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "rouge_score_version": safe_version("rouge-score"),
    "pydantic_version": safe_version("pydantic"),
    "git_commit": safe_git_commit(PROJECT_ROOT),
    "git_dirty": is_git_dirty(PROJECT_ROOT),
}

print("PROJECT_ROOT:", PROJECT_ROOT)
environment_snapshot


PROJECT_ROOT: C:\Users\OS\OneDrive\Desktop\Text_Summarization


{'python_version': '3.10.20',
 'platform': 'Windows-10-10.0.26200-SP0',
 'pandas_version': '2.3.3',
 'rouge_score_version': '0.1.2',
 'pydantic_version': '2.12.5',
 'git_commit': 'c5f2468d3b654fd7e944f9edf35afe1d504e936c',
 'git_dirty': True}

## 2. Configuration

Record these settings explicitly when transferring results into the thesis report.

In [58]:
NOTEBOOK_SCHEMA_VERSION = "tfidf_phase1_analysis_v1"
PROTOCOL_VERSION_EXPECTED = "phase0_v2"
TARGET_SPLIT = "validation"  # validation / test
SEED = 42
TOP_K_CANDIDATES = [2, 3, 4, 5]
SUBSET_LIMIT = 200
ARTICLE_CHAR_THRESHOLD = 1200  # set to None to use all eligible samples

random.seed(SEED)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "vietnews"
MANIFEST_PATH = PROCESSED_DIR / "dataset_manifest.json"
SPLIT_PATH = PROCESSED_DIR / f"{TARGET_SPLIT}.jsonl"

config_snapshot = {
    "notebook_schema_version": NOTEBOOK_SCHEMA_VERSION,
    "protocol_version_expected": PROTOCOL_VERSION_EXPECTED,
    "target_split": TARGET_SPLIT,
    "seed": SEED,
    "top_k_candidates": TOP_K_CANDIDATES,
    "subset_limit": SUBSET_LIMIT,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
}
config_snapshot


{'notebook_schema_version': 'tfidf_phase1_analysis_v1',
 'protocol_version_expected': 'phase0_v2',
 'target_split': 'validation',
 'seed': 42,
 'top_k_candidates': [2, 3, 4, 5],
 'subset_limit': 200,
 'article_char_threshold': 1200}

## 3. Load The Processed Dataset And Freeze The Evaluation Subset

This step reads only the processed split and verifies the manifest protocol version before benchmarking.

In [59]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Processed split not found: {SPLIT_PATH}")

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

protocol_version = manifest.get("protocol_version")
if protocol_version != PROTOCOL_VERSION_EXPECTED:
    raise RuntimeError(
        f"Protocol version mismatch: expected {PROTOCOL_VERSION_EXPECTED!r}, got {protocol_version!r}"
    )

rows: list[dict] = []
with SPLIT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("The processed split is empty.")

required_cols = ["guid", "title", "article", "reference_summary", "meta"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in the processed split: {missing_cols}")

df["article_char_len"] = df["meta"].map(lambda m: (m or {}).get("article_char_len", 0))
df["reference_char_len"] = df["meta"].map(lambda m: (m or {}).get("reference_summary_char_len", 0))
df = df[df["article"].fillna("").str.strip() != ""].copy()
df = df[df["reference_summary"].fillna("").str.strip() != ""].copy()

if ARTICLE_CHAR_THRESHOLD is not None:
    df = df[df["article_char_len"] >= ARTICLE_CHAR_THRESHOLD].copy()

subset_df = df.sample(frac=1.0, random_state=SEED).head(SUBSET_LIMIT).copy()
subset_df.reset_index(drop=True, inplace=True)

subset_overview = {
    "loaded_rows": int(len(rows)),
    "eligible_rows_after_filter": int(len(df)),
    "subset_rows": int(len(subset_df)),
    "target_split": TARGET_SPLIT,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
}
subset_overview


{'loaded_rows': 22184,
 'eligible_rows_after_filter': 18884,
 'subset_rows': 200,
 'target_split': 'validation',
 'article_char_threshold': 1200}

## 4. Qualitative Inspection On One Frozen Example

This section is used only to explain how TF-IDF ranks and selects sentences.

In [60]:
demo_row = subset_df.iloc[0].to_dict()
demo_processed = process_from_text(str(demo_row.get("article", "")))

inspect_rows: list[dict] = []
for top_k in TOP_K_CANDIDATES:
    selected_sentences, engine_meta = summarize_with_tfidf(
        demo_processed.sentences,
        max_sentences=top_k,
        ratio=None,
    )
    selected_indices = set(engine_meta.get("selected_indices", []))
    for rank, item in enumerate(
        sorted(engine_meta.get("sentence_scores", []), key=lambda x: x["score"], reverse=True),
        start=1,
    ):
        idx = item["index"]
        inspect_rows.append(
            {
                "top_k": top_k,
                "rank_by_score": rank,
                "sentence_index": idx,
                "score": item["score"],
                "is_selected": idx in selected_indices,
                "sentence": demo_processed.sentences[idx],
            }
        )

inspect_df = pd.DataFrame(inspect_rows)
inspect_df.head(20)


,top_k,rank_by_score,sentence_index,score,is_selected,sentence
0,2,1,2,2.537019,True,"Theo anh Đức , thời_gian qua , đã đưa tin , phản ánh nhiều vụ việc phức tạp , tiêu_cực , ảnh hưởng đến quyền lợi của một số cá nhân , tổ_chức nên nghi_ngờ bị kẻ xấu đe_đoạ , trả_thù ."
1,2,2,1,2.232650,True,"Trước đó , chiều 4/5 anh Đức liên tục nhận được điện thoại từ 4 số lạ gọi đến với nội dung đe dọa đến bản thân anh và gia đình của anh ."
2,2,3,5,2.176987,False,"Ngay sau đó , một tổ công_tác của phòng Cảnh_sát hình_sự được cử xuống TP. Hồ_Chí_Minh để xác minh , và mời đối tượng Nhi về trụ sở Công_an tỉnh Đắk Lắk để làm việc ."
3,2,4,0,2.161786,False,"Tối 13/5 , thông_tin từ phòng Cảnh_sát hình_sự , Công_an tỉnh Đắk_Lắk cho biết , chiều cùng ngày đơn_vị đã đưa đối_tượng Nguyễn Hoàng Quân_Nhi , 23 tuổi , nhân_viên công_ty đòi nợ thuê ở TP. HCM gặp mặt , xin lỗi anh Nguyễn Sỹ Đức ( PV ..."
4,2,5,4,2.148628,False,"Sau khi tiếp_nhận trình_báo , bằng các biện_pháp nghiệp lực_lượng Công_an tỉnh Đắk_Lắk xác_định được đối tượng đe_doạ anh Đức là Nguyễn_Hoàng_Quân_Nhi , hiện đang ở quận 11 , TP. Hồ_Chí_Minh ."
5,2,6,8,2.110982,False,"Sau đó , Nhi dùng nhiều số điện_thoại của công_ty và cá_nhân gọi vào số máy của anh Đức đe dọa , nhằm mục đích đòi nợ , nhưng nhầm người ."
6,2,7,7,2.095659,False,"Ngày 4/5 , Nhi nhận nhiệm vụ đòi nợ một người có tên Nguyễn Minh Đức , nhưng quá_trình tìm số điện_thoại đã nhầm lẫn sang số điện_thoại của anh Nguyễn Sỹ Đức ."
7,2,8,9,2.057958,False,"Tại cơ_quan công_an , Nhi đã xin_lỗi anh Đức và mong anh thông_cảm vì sự nhầm_lẫn của mình ."
8,2,9,10,1.976046,False,Đối_tượng Nhi đã nhiều lần gọi điện đe_doạ phóng_viên Nguyễn_Sỹ_Đức .
9,2,10,6,1.959396,False,"Tại cơ_quan công_an , đối_tượng Nhi khai nhận mình là nhân_viên một công_ty đòi nợ thuê ở TP. Hồ_Chí_Minh ."


## 5. Run The Benchmark

Each sample is summarized with TF-IDF and then evaluated using ROUGE, latency, compression ratio, and repetition rate.

In [61]:
def summarize_article(article: str, top_k: int) -> tuple[str, dict, float, str]:
    processed = process_from_text(article)
    t0 = time.perf_counter()
    selected_sentences, engine_meta = summarize_with_tfidf(
        processed.sentences,
        max_sentences=top_k,
        ratio=None,
    )
    summarizer_core_latency_sec = time.perf_counter() - t0
    summary = " ".join(sentence.strip() for sentence in selected_sentences if sentence.strip())
    return summary, engine_meta, summarizer_core_latency_sec, processed.cleaned_text


records: list[dict] = []
for top_k in TOP_K_CANDIDATES:
    for row in subset_df.to_dict(orient="records"):
        article = str(row.get("article", ""))
        reference = str(row.get("reference_summary", ""))
        if not article.strip() or not reference.strip():
            continue

        predicted_summary, engine_meta, summarizer_core_latency_sec, cleaned_article = summarize_article(article, top_k)
        bundle = evaluator.evaluate_one(
            source_text=cleaned_article,
            reference_summary=reference,
            predicted_summary=predicted_summary,
            latency_sec=summarizer_core_latency_sec,
            extra={
                "guid": row.get("guid"),
                "top_k": top_k,
                "article_char_len": row.get("article_char_len"),
                "reference_char_len": row.get("reference_char_len"),
                "engine": engine_meta.get("engine", "tfidf"),
            },
        )
        rec = bundle.as_dict()
        rec["summarizer_core_latency_sec"] = rec.pop("latency_sec")
        records.append(rec)

bench_df = pd.DataFrame(records)
if bench_df.empty:
    raise RuntimeError("No valid samples are available for benchmarking.")

summary_df = (
    bench_df.groupby("top_k", as_index=False)
    .agg(
        n=("top_k", "count"),
        rouge1_f=("rouge1_f", "mean"),
        rouge2_f=("rouge2_f", "mean"),
        rougeL_f=("rougeL_f", "mean"),
        summarizer_core_latency_sec=("summarizer_core_latency_sec", "mean"),
        compression_ratio=("compression_ratio", "mean"),
        repetition_rate=("repetition_rate", "mean"),
    )
    .sort_values("top_k")
    .reset_index(drop=True)
)
summary_df


,top_k,n,rouge1_f,rouge2_f,rougeL_f,summarizer_core_latency_sec,compression_ratio,repetition_rate
0,2,200,0.476567,0.138918,0.267045,0.000468,0.117981,0.010736
1,3,200,0.452707,0.162854,0.264224,0.000480,0.185369,0.014293
2,4,200,0.399962,0.164417,0.243029,0.000484,0.249858,0.020934
3,5,200,0.350163,0.168769,0.227097,0.000483,0.317199,0.025651


## 6. Select The Best Hyperparameter And Summarize The Findings

In [62]:
selection_view_df = summary_df.copy()
selection_view_df["rank_rouge1"] = selection_view_df["rouge1_f"].rank(ascending=False, method="min")
selection_view_df["rank_rouge2"] = selection_view_df["rouge2_f"].rank(ascending=False, method="min")
selection_view_df["rank_rougeL"] = selection_view_df["rougeL_f"].rank(ascending=False, method="min")
selection_view_df["rank_compression"] = selection_view_df["compression_ratio"].rank(ascending=True, method="min")
selection_view_df["rank_repetition"] = selection_view_df["repetition_rate"].rank(ascending=True, method="min")
selection_view_df["rank_latency"] = selection_view_df["summarizer_core_latency_sec"].rank(ascending=True, method="min")

selection_view_df["weighted_rank_score"] = (
    0.30 * selection_view_df["rank_rouge1"]
    + 0.20 * selection_view_df["rank_rouge2"]
    + 0.25 * selection_view_df["rank_rougeL"]
    + 0.15 * selection_view_df["rank_compression"]
    + 0.05 * selection_view_df["rank_repetition"]
    + 0.05 * selection_view_df["rank_latency"]
)

selection_view_df = selection_view_df.sort_values(["weighted_rank_score", "rouge1_f"], ascending=[True, False]).reset_index(drop=True)
recommended_row = selection_view_df.iloc[0].to_dict()

analysis_selected_top_k = int(recommended_row["top_k"])
analysis_top_k_rationale = (
    "Notebook recommendation from weighted-rank selection for exploratory analysis. "
    "Use script-generated official artifacts for frozen reporting decisions."
)

conclusion = {
    "recommended_top_k_by_weighted_rank": int(recommended_row["top_k"]),
    "analysis_selected_top_k": analysis_selected_top_k,
    "analysis_top_k_rationale": analysis_top_k_rationale,
    "tradeoff_note": (
        "Weighted rank balances quality (ROUGE-1/2/L) and efficiency (compression, repetition, latency)."
    ),
}
conclusion


{'recommended_top_k_by_weighted_rank': 2,
 'analysis_selected_top_k': 2,
 'analysis_top_k_rationale': 'Notebook recommendation from weighted-rank selection for exploratory analysis. Use script-generated official artifacts for frozen reporting decisions.',
 'tradeoff_note': 'Weighted rank balances quality (ROUGE-1/2/L) and efficiency (compression, repetition, latency).'}

## 7. Save Benchmark Artifacts

This cell is for local notebook exports and analysis convenience.
For canonical official artifacts used in reproducible reporting, use `scripts/generate_official_validation_artifacts.py`.

In [63]:
out_dir = PROJECT_ROOT / "notebooks" / "results" / "analysis"
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = out_dir / f"tfidf_phase1_topk_summary_analysis_{ts}.csv"
detail_path = out_dir / f"tfidf_phase1_topk_detail_analysis_{ts}.csv"
report_path = out_dir / f"tfidf_phase1_topk_report_analysis_{ts}.json"

summary_df.to_csv(summary_path, index=False, encoding="utf-8")
bench_df.to_csv(detail_path, index=False, encoding="utf-8")

report_payload = {
    "report_schema_version": NOTEBOOK_SCHEMA_VERSION,
    "notebook": "02_tfidf_experiment.ipynb",
    "run_purpose": "analysis_recipe_notebook",
    "protocol_version": protocol_version,
    "target_split": TARGET_SPLIT,
    "seed": SEED,
    "top_k_candidates": TOP_K_CANDIDATES,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
    "subset_limit": SUBSET_LIMIT,
    "subset_rows": int(len(subset_df)),
    "subset_guid_sample": subset_df["guid"].head(20).tolist(),
    "environment": environment_snapshot,
    "config_snapshot": config_snapshot,
    "recommended_top_k_by_weighted_rank": int(recommended_row["top_k"]),
    "analysis_selected_top_k": int(recommended_row["top_k"]),
    "benchmark_notes": {
        "latency_label": "summarizer_core_latency_sec",
        "latency_scope": "Measured around summarize_with_tfidf only; excludes process_from_text",
        "selection_method": "Weighted rank over ROUGE-1/2/L, compression_ratio, repetition_rate, latency",
        "official_artifact_source": "scripts/generate_official_validation_artifacts.py",
    },
    "summary_csv": str(summary_path),
    "detail_csv": str(detail_path),
    "manifest_path": str(MANIFEST_PATH),
    "split_path": str(SPLIT_PATH),
}

report_path.write_text(json.dumps(report_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved TF-IDF notebook analysis artifacts:")
print(summary_path)
print(detail_path)
print(report_path)
report_payload


Saved TF-IDF notebook analysis artifacts:
C:\Users\OS\OneDrive\Desktop\Text_Summarization\notebooks\results\analysis\tfidf_phase1_topk_summary_analysis_20260426_031904.csv
C:\Users\OS\OneDrive\Desktop\Text_Summarization\notebooks\results\analysis\tfidf_phase1_topk_detail_analysis_20260426_031904.csv
C:\Users\OS\OneDrive\Desktop\Text_Summarization\notebooks\results\analysis\tfidf_phase1_topk_report_analysis_20260426_031904.json


{'report_schema_version': 'tfidf_phase1_analysis_v1',
 'notebook': '02_tfidf_experiment.ipynb',
 'run_purpose': 'analysis_recipe_notebook',
 'protocol_version': 'phase0_v2',
 'target_split': 'validation',
 'seed': 42,
 'top_k_candidates': [2, 3, 4, 5],
 'article_char_threshold': 1200,
 'subset_limit': 200,
 'subset_rows': 200,
 'subset_guid_sample': [494,
  8563,
  55,
  11152,
  5005,
  10389,
  7706,
  3927,
  14906,
  10408,
  14913,
  14438,
  1241,
  19076,
  13998,
  4850,
  19760,
  2581,
  5590,
  14528],
 'environment': {'python_version': '3.10.20',
  'platform': 'Windows-10-10.0.26200-SP0',
  'pandas_version': '2.3.3',
  'rouge_score_version': '0.1.2',
  'pydantic_version': '2.12.5',
  'git_commit': 'c5f2468d3b654fd7e944f9edf35afe1d504e936c',
  'git_dirty': True},
 'config_snapshot': {'notebook_schema_version': 'tfidf_phase1_analysis_v1',
  'protocol_version_expected': 'phase0_v2',
  'target_split': 'validation',
  'seed': 42,
  'top_k_candidates': [2, 3, 4, 5],
  'subset_lim

## 8. How To Read The Results

- `summary_df`: aggregate ROUGE, summarizer-core latency, compression ratio, and repetition rate for each `top_k`.
- `bench_df`: per-sample detail table, suitable for later error analysis.
- `report_payload`: notebook-side metadata snapshot for analysis context.
- Official artifacts and reproducible report payload are generated by `scripts/generate_official_validation_artifacts.py`.
- Error analysis artifact is generated by `scripts/generate_short_error_analysis.py` to keep this notebook focused on benchmark exploration.
